# Deep Learning Project — WikiArt Image Classification

In [ ]:
import zipfile
import os
import shutil
import random
import matplotlib.pyplot as plt
import numpy as np

dataset_path = "data/raw/wikiart"

if not os.path.exists(dataset_path):
    os.makedirs("data/raw", exist_ok=True)
    with zipfile.ZipFile("wikiart.zip", "r") as z:
        z.extractall("data/raw")
    print("Dataset extracted.")
else:
    print("Dataset already extracted.")

Dataset extracted.


## Data Split

Split the raw dataset into **train / validation / test** folders (70 / 15 / 15) with stratified sampling per artist.  
The split only runs if the target directories do not already exist.

In [ ]:
train_dir = "data/train"
val_dir   = "data/val"
test_dir  = "data/test"

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO = 0.15  (remainder)

if not all(os.path.exists(d) for d in [train_dir, val_dir, test_dir]):
    random.seed(SEED)

    classes = sorted(os.listdir(dataset_path))
    for cls in classes:
        src = os.path.join(dataset_path, cls)
        images = sorted(os.listdir(src))
        random.shuffle(images)

        n = len(images)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        splits = {
            train_dir: images[:n_train],
            val_dir:   images[n_train:n_train + n_val],
            test_dir:  images[n_train + n_val:],
        }

        for split_dir, file_list in splits.items():
            dst = os.path.join(split_dir, cls)
            os.makedirs(dst, exist_ok=True)
            for fname in file_list:
                shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))

    print("Data split complete.")
    for name, d in [("Train", train_dir), ("Val", val_dir), ("Test", test_dir)]:
        total = sum(len(os.listdir(os.path.join(d, c))) for c in os.listdir(d))
        print(f"  {name}: {total} images")
else:
    print("Split directories already exist — skipping.")
    for name, d in [("Train", train_dir), ("Val", val_dir), ("Test", test_dir)]:
        total = sum(len(os.listdir(os.path.join(d, c))) for c in os.listdir(d))
        print(f"  {name}: {total} images")

In [ ]:
# Quick sanity check: show one sample per split
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, d) in zip(axes, [("Train", train_dir), ("Val", val_dir), ("Test", test_dir)]):
    cls = sorted(os.listdir(d))[0]
    img_path = os.path.join(d, cls, os.listdir(os.path.join(d, cls))[0])
    ax.imshow(plt.imread(img_path))
    ax.set_title(f"{name} — {cls}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Baseline CNN Model

A minimal Convolutional Neural Network to establish a baseline:
- 3 Conv2D + MaxPooling blocks
- Global Average Pooling → Dense → Softmax (23 classes)
- Images resized to **128 × 128** for fast iteration

In [ ]:
import keras
from keras import layers

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 23

# --- Data generators with basic rescaling ---
train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
)

class_names = train_ds.class_names
print(f"Classes ({len(class_names)}): {class_names}")

In [ ]:
# Prefetch for performance
AUTOTUNE = keras.backend.backend()  # just need tf.data.AUTOTUNE
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# --- Build the model ---
model = keras.Sequential([
    # Rescale pixel values to [0, 1]
    layers.Rescaling(1.0 / 255, input_shape=(*IMG_SIZE, 3)),

    # Block 1
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    # Block 2
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    # Block 3
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    # Head
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

### Training

In [ ]:
EPOCHS = 20

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
)

### Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"], label="train")
ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history.history["accuracy"], label="train")
ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

### Test Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Collect predictions
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Baseline CNN")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()